# Occurrence Records of African Bats with Taxonomic, Geographic, and Temporal Annotations Exploration with `mlcroissant`
This notebook demonstrates loading and exploring a dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. We'll load the Croissant schema, access the metadata, inspect the available record sets (tables), and perform some example analyses using the unique `@id` identifiers mandated by the Croissant standard.

### Dataset Source
The dataset follows the [Croissant specification](https://mlcommons.org/croissant), and its schema is available at:

`https://sen.science/doi/10.71728/senscience.swcc-jqh9/fair2.json`

In [ ]:
# Install mlcroissant library if it is not already installed
!pip install mlcroissant

## 1. Data Loading
Load the Croissant metadata and records using `mlcroissant`. We'll also display a summary of the dataset.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.swcc-jqh9/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print high level metadata
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")

# Optionally print additional metadata
print(f"\nPublished: {getattr(meta, 'datePublished', 'Unknown')}")
print(f"Identifier: {getattr(meta, 'identifier', 'Unknown')}")
print(f"Temporal coverage: {getattr(meta, 'temporalCoverage', 'Unknown')}")
print(f"Spatial coverage: {getattr(meta, 'spatialCoverage', 'Unknown')}")
print(f"License: {getattr(meta, 'license', 'Unknown')}")

## 2. Data Overview
Let's enumerate the available record sets, the fields (columns) within each, and their Croissant `@id` values. These ids are essential for precise referencing and extraction.

In [ ]:
# List all record sets, their fields, and Croissant @id fields
print("Available record sets:")
for record_set in dataset.record_sets:
    print(f"- Record set name: {record_set.name} | @id: {record_set.id}")
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - {field.name} (@id: {field.id}, type: {field.data_type})")
    print()

## 3. Data Extraction
Let's programmatically extract the data from all record sets. The code below creates a dataframe for each available record set (referenced by `@id`) and shows the first few records from a primary or main one.

In [ ]:
# Extract all dataframes, referenced by record set @id
record_sets = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Show available record sets
print("DataFrames loaded for these record sets:")
for record_set_id in dataframes:
    print(f"- {record_set_id} (shape: {dataframes[record_set_id].shape})")

# For the main record set, show column ids and preview
main_record_set_id = record_sets[0] if record_sets else None
if main_record_set_id is not None:
    print(f"\nColumns (@id) of '{main_record_set_id}':\n{dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets found.")

## 4. Exploratory Data Analysis (EDA)
We demonstrate basic exploration using field `@id`s (as shown above). We'll select a numeric field, filter records, normalize, and optionally group by a categorical field, all identified by their `@id`.

In [ ]:
# -- Choose the first record set as the main one --
main_record_set_id = record_sets[0] if record_sets else None

# Inspect its columns to find likely numeric fields
if main_record_set_id is None:
    print("No main record set to analyze.")
else:
    df = dataframes[main_record_set_id]
    columns = df.columns.tolist()
    print(f"Available fields in '{main_record_set_id}':\n{columns}")
    # Example: Use longitude or latitude if available for thresholding
    # (Field IDs to use will depend on the dataset's schema. Below is generic; adjust as needed.)
    numeric_candidates = [col for col in columns if 'latitude' in col.lower() or 'longitude' in col.lower() or 'year' in col.lower()]

    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
        print(f"\nUsing numeric field: '{numeric_field_id}' for further analysis.")
        # Convert to numeric, handle NaN
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = df[numeric_field_id].quantile(0.5)  # median as threshold
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"\nFiltered rows with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize
        if filtered_df[numeric_field_id].std() > 0:
            filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
            print(f"\nNormalized '{numeric_field_id}':")
            display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        else:
            print(f"\nField {numeric_field_id} has zero variance after filtering; skipping normalization.")

        # Try grouping by a categorical field (e.g., 'country', 'family', etc. by @id)
        group_candidates = [col for col in columns if 'country' in col.lower() or 'family' in col.lower() or 'genus' in col.lower()]
        if group_candidates:
            group_field = group_candidates[0]
            print(f"\nGrouping results by '{group_field}':")
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index().sort_values(by=numeric_field_id, ascending=False)
            display(grouped_df.head())
        else:
            print("No categorical field found to group by.")
    else:
        print("No suitable numeric field found for EDA.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field and, if available, the counts or averages by the selected group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize only if data exists from above
if main_record_set_id is not None and numeric_candidates:
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))

    # Histogram of numeric field (using non-null values)
    sns.histplot(df[numeric_field_id].dropna(), bins=30, ax=ax[0], color='steelblue')
    ax[0].set_title(f"Distribution of {numeric_field_id}")
    ax[0].set_xlabel(numeric_field_id)
    ax[0].set_ylabel('Count')

    # Bar plot by group (if grouping done)
    if 'grouped_df' in locals() and not grouped_df.empty:
        top_groups = grouped_df.head(10)
        sns.barplot(y=top_groups[group_field], x=top_groups[numeric_field_id], ax=ax[1], palette='viridis')
        ax[1].set_title(f"Top 10 groups by mean {numeric_field_id}")
        ax[1].set_xlabel(f"Mean {numeric_field_id}")
        ax[1].set_ylabel(group_field)
    else:
        ax[1].axis('off')

    plt.tight_layout()
    plt.show()
else:
    print("Insufficient numeric data to plot.")

## 6. Conclusion

- The dataset comprises occurrence records of African bats, with detailed taxonomy, location, and temporal annotations.
- We accessed all data elements using their Croissant `@id` identifiers for precision and reproducibility.
- Simple exploratory analysis and plotting reveals overall numeric distributions and patterns (e.g., latitude or year) and shows how to aggregate or compare by categorical attributes such as country or family.

You can extend this notebook with domain-specific analyses, e.g. geo-mapping occurrences, richer statistics, or linkage to species conservation status, always referencing data elements by their `@id`.